[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabrielbelo2007/RP_PIGuard/blob/main/Notebooks/second_train/train.ipynb)

# PIGuard - Estudo de Ablação com MOF (Fase 2: Scratch Retrain & Finetuning)

Este notebook executa todos os treinamentos da **Tabela de Ablação (Tabela 2)** do artigo PIGuard utilizando aceleração por GPU no Google Colab.

---

### Mapeamento dos Cenários e das 3 Vertentes

| Design de Treinamento (Artigo) | Métricas Artigo (Overdef / Benign / Mal / Avg) | Vertente 1: Original (Artigo) | Vertente 2: Oficial (Re-gerada) | Vertente 3: Reproduzida |
| :--- | :---: | :--- | :--- | :--- |
| **Basic dataset training** | 75.22 / 78.53 / 70.17 / 74.64 | `train_cleaned_original.json` *(Fase 1)* | `train_cleaned_original.json` *(Fase 1)* | `train_cleaned_original.json` *(Fase 1)* |
| **+ Data-centric augment** | 64.31 / 81.36 / 75.95 / 73.87 | `train_augmented_original.json` *(Fase 1)* | `train_augmented_original.json` *(Fase 1)* | `train_augmented_rp.json` *(Fase 1)* |
| **Basic + MOF scratch retrain** | **89.38** / 84.73 / 71.57 / 81.89 | `MOF/train_clean_mof_original.json` | `MOF/train_clean_mof_semi_rp.json` | `MOF/train_clean_mof_rp.json` |
| **Basic + Aug + MOF finetuning** | 68.14 / 82.11 / 74.81 / 75.02 | `MOF/MOF_original.json` *(resume oficial)* | `MOF/MOF_semi_rp.json` *(resume oficial)* | `MOF/MOF_rp.json` *(resume reprod)* |
| **Basic + Aug + MOF scratch retrain** | 87.32 / **85.74** / **77.39** / **83.48** | `datasets/train.json` *(Original)* | `MOF/train_augmented_mof_semi_rp.json` | `MOF/train_augmented_mof_rp.json` |

---

### Proteções de Engenharia e Boas Práticas:
1. **Prevenção de Colapso de Classe no Fine-Tuning (`MOF_rp.json` 100% Benigno)**:
   - `--save_step 2` e `--warmup 2`: Como cada época tem apenas 24 passos, avaliações ultra-frequentes capturam o modelo nos passos iniciais antes da perda de sensibilidade à classe maliciosa.
   - Trava anti-colapso em `train.py` (`injection_accuracy > 0.5` e `total_accuracy > save_thres`).
2. **Persistência Segura no Google Drive**: Checkpoints salvos diretamente em `/content/drive/MyDrive/RP_PIGuard_outputs`.
3. **Pulo Inteligente (`FORCE_RETRAIN = False`)**: Modelos já treinados e persistidos no Drive são pulados para economizar horas/créditos de GPU.
4. **Desligamento Automático da VM**: Libera a GPU assim que todos os 9 treinamentos terminarem.

In [ ]:
# Célula 1: Verificação da GPU
!nvidia-smi

import torch
print("\n--- Informações de Hardware e PyTorch ---")
print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


### Setup do Ambiente: Google Drive, Repositório e Dependências

In [ ]:
# Célula 2: Montagem do Google Drive, Repositório e Dependências
import os
import sys
import shutil

# 1. Montagem do Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/RP_PIGuard_outputs"
EXPERIMENT_NAMES = [
    # Vertente 1: Datasets Originais do Artigo
    "train_clean_mof_original",
    "train_augmented_mof_finetuning_original",
    "train_augmented_mof_original",
    # Vertente 2: Vertente Oficial (Re-gerada)
    "train_clean_mof_oficial",
    "train_augmented_mof_finetuning_oficial",
    "train_augmented_mof_oficial",
    # Vertente 3: Vertente Reproduzida
    "train_clean_mof",
    "train_augmented_mof_finetuning",
    "train_augmented_mof"
]

for exp in EXPERIMENT_NAMES:
    os.makedirs(f"{DRIVE_OUTPUT_DIR}/{exp}", exist_ok=True)
print(f"Diretórios de persistência configurados no Google Drive: {DRIVE_OUTPUT_DIR}")

# 2. Clonagem ou sincronização do repositório
REPO_DIR = "/content/RP_PIGuard"
if not os.path.exists(REPO_DIR):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass

    if not token and os.path.exists("/content/drive/MyDrive/RP_PIGuard"):
        print("Copiando repositório existente do Google Drive para o SSD local da VM...")
        shutil.copytree("/content/drive/MyDrive/RP_PIGuard", REPO_DIR)
    else:
        repo_url = f"https://{token}@github.com/gabrielbelo2007/RP_PIGuard.git" if token else "https://github.com/gabrielbelo2007/RP_PIGuard.git"
        print(f"Clonando repositório em {REPO_DIR}...")
        !git clone {repo_url} {REPO_DIR}
else:
    print(f"Repositório já presente em {REPO_DIR}. Atualizando branch main...")
    !cd {REPO_DIR} && git pull origin main || true

# 3. Mudar para a raiz do repositório
%cd {REPO_DIR}

# 4. Instalar dependências necessárias
!pip install -q "transformers>=4.44.0,<5.0.0" "datasets>=2.21.0" sentencepiece protobuf tiktoken pyyaml nltk tqdm accelerate

# 5. Controle de retreinamento:
# Se False, pula treinamentos cujos modelos (best_model.pth) já existam no Google Drive
FORCE_RETRAIN = False
print(f"\nFORCE_RETRAIN configurado como: {FORCE_RETRAIN}")

# Adiciona ao sys.path e altera cwd explicitamente
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 6. Verificação dos datasets e checkpoints base
print("\n--- Verificação de Datasets e Checkpoints Base ---")
def check_file(path, desc):
    exists = os.path.exists(path)
    status = "OK" if exists else "AUSENTE"
    print(f"[{status}] {desc}: {path}")
    return exists

print("Datasets MOF:")
for ds in [
    "MOF/train_clean_mof_original.json", "MOF/MOF_original.json", "datasets/train.json",
    "MOF/train_clean_mof_semi_rp.json", "MOF/MOF_semi_rp.json", "MOF/train_augmented_mof_semi_rp.json",
    "MOF/train_clean_mof_rp.json", "MOF/MOF_rp.json", "MOF/train_augmented_mof_rp.json"
]:
    check_file(ds, ds.split('/')[-1])

print("\nCheckpoints Base para Fine-Tuning (Fase 1):")
def find_checkpoint(candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

base_oficial_candidates = [
    f"{DRIVE_OUTPUT_DIR}/train_augmented_original/best_model.pth",
    f"{DRIVE_OUTPUT_DIR}/train_augmented_rp/best_model.pth",
    "checkpoints/train_augmented_original/best_model.pth"
]
base_reprod_candidates = [
    f"{DRIVE_OUTPUT_DIR}/train_augmented_rp/best_model.pth",
    "checkpoints/train_augmented_rp/best_model.pth"
]

base_oficial = find_checkpoint(base_oficial_candidates)
base_reprod = find_checkpoint(base_reprod_candidates)
print(f"Checkpoint Base Oficial: {base_oficial}")
print(f"Checkpoint Base Reproduzido: {base_reprod}")


### Função Helper de Execução e Persistência de Treinamentos

In [ ]:
# Função para execução padronizada de treinamento, log e sincronização com Drive
def run_mof_experiment(name, train_set, save_step=100, warmup=100, epochs=3, resume_ckpt=None):
    import os
    import shutil
    
    drive_target = f"{DRIVE_OUTPUT_DIR}/{name}"
    drive_best = f"{drive_target}/best_model.pth"
    
    if os.path.exists(drive_best) and not FORCE_RETRAIN:
        print(f"\n[SKIP] Modelo '{name}' já existe no Google Drive em: {drive_best}")
        print("Pulando para economizar tempo e recursos de GPU. (Defina FORCE_RETRAIN=True para reexecutar).")
        return
        
    print(f"\n{'='*75}")
    print(f"🚀 INICIANDO EXPERIMENTO: {name}")
    print(f"📁 Dataset: {train_set}")
    print(f"⚙️ Configuração: save_step={save_step}, warmup={warmup}, epochs={epochs}")
    if resume_ckpt:
        print(f"🔄 Resumindo pesos de: {resume_ckpt}")
        if not os.path.exists(resume_ckpt):
            raise FileNotFoundError(f"Checkpoint base não encontrado: {resume_ckpt}. Execute o treinamento base primeiro!")
    print(f"{'='*75}\n")
    
    os.makedirs(f"logs/{name}", exist_ok=True)
    os.makedirs(f"checkpoints/{name}", exist_ok=True)
    
    cmd = [
        "python", "train.py",
        "--name", name,
        "--train_set", train_set,
        "--valid_set", "datasets/valid.json",
        "--dataset_root", "datasets",
        "--checkpoint_path", f"checkpoints/{name}",
        "--logs", f"logs/{name}",
        "--batch_size", "32",
        "--eval_batch_size", "32",
        "--save_step", str(save_step),
        "--warmup", str(warmup),
        "--epochs", str(epochs)
    ]
    if resume_ckpt:
        cmd.extend(["--resume", resume_ckpt])
        
    cmd_str = " ".join(cmd)
    ret = os.system(cmd_str)
    if ret != 0:
        raise RuntimeError(f"Erro durante o treinamento de {name} (código de saída: {ret})")
        
    local_best = f"checkpoints/{name}/best_model.pth"
    local_log = f"logs/{name}/log_{name}.txt"
    if os.path.exists(local_best):
        print(f"\n💾 Copiando {local_best} para {drive_target}...")
        shutil.copy(local_best, f"{drive_target}/best_model.pth")
        if os.path.exists(local_log):
            shutil.copy(local_log, f"{drive_target}/log_{name}.txt")
        print(f"✅ Checkpoint salvo com sucesso no Google Drive: {drive_target}/best_model.pth")
        
        # Limpar checkpoints intermediários de época para poupar espaço em disco na VM
        for fname in os.listdir(f"checkpoints/{name}"):
            if fname.startswith("epoch_") and fname.endswith(".pth"):
                try:
                    os.remove(os.path.join(f"checkpoints/{name}", fname))
                except Exception:
                    pass
    else:
        print(f"⚠️ Aviso: {local_best} não foi gerado.")


## Vertente 1: Datasets Originais do Artigo
Nesta vertente, reproduzimos os cenários do artigo utilizando:
- **`train_clean_mof_original.json`** (76.300 amostras: base limpa original + 762 mitigações originais).
- **`MOF_original.json`** (762 amostras de mitigação originais com `source == 'over-defense'`).
- **`datasets/train.json`** (76.735 amostras: base limpa original + 435 aumentados + 762 mitigações originais).

### Experimento 1: `train_clean_mof_original` (Basic + MOF scratch retrain)
- **Dataset**: `MOF/train_clean_mof_original.json` (76.300 amostras)
- **Modo**: Scratch Retrain (`--save_step 100 --warmup 100 --epochs 3`)
- **Meta do Artigo**: Over-defense 89.38% | Benign 84.73% | Malicious 71.57% | Overall 81.89%

In [ ]:
# Célula 3: Treinamento 1 - train_clean_mof_original
run_mof_experiment(
    name="train_clean_mof_original",
    train_set="MOF/train_clean_mof_original.json",
    save_step=100,
    warmup=100,
    epochs=3,
    resume_ckpt=None
)


### Experimento 2: `train_augmented_mof_finetuning_original` (Basic + Aug + MOF finetuning)
- **Dataset**: `MOF/MOF_original.json` (762 amostras de mitigação originais)
- **Modo**: Fine-Tuning com `--resume` a partir do checkpoint de `train_augmented_original`
- **Safeguard Anti-Colapso**: `--save_step 2 --warmup 2` e validação imediata no step 1
- **Meta do Artigo**: Over-defense 68.14% | Benign 82.11% | Malicious 74.81% | Overall 75.02%

In [ ]:
# Célula 4: Treinamento 2 - train_augmented_mof_finetuning_original
base_ckpt = find_checkpoint([
    f"{DRIVE_OUTPUT_DIR}/train_augmented_original/best_model.pth",
    "checkpoints/train_augmented_original/best_model.pth",
    f"{DRIVE_OUTPUT_DIR}/train_augmented_rp/best_model.pth"
])
print(f"Utilizando checkpoint base: {base_ckpt}")

run_mof_experiment(
    name="train_augmented_mof_finetuning_original",
    train_set="MOF/MOF_original.json",
    save_step=2,
    warmup=2,
    epochs=3,
    resume_ckpt=base_ckpt
)


### Experimento 3: `train_augmented_mof_original` (Basic + Aug + MOF scratch retrain)
- **Dataset**: `datasets/train.json` (76.735 amostras originais completas do artigo)
- **Modo**: Scratch Retrain (`--save_step 100 --warmup 100 --epochs 3`)
- **Meta do Artigo**: Over-defense 87.32% | Benign 85.74% | Malicious 77.39% | Overall 83.48%

In [ ]:
# Célula 5: Treinamento 3 - train_augmented_mof_original (Dataset Original Completo)
run_mof_experiment(
    name="train_augmented_mof_original",
    train_set="datasets/train.json",
    save_step=100,
    warmup=100,
    epochs=3,
    resume_ckpt=None
)


## Vertente 2: Vertente Oficial (Re-gerada via Trigger Words Oficiais)
Nesta vertente, utilizamos as 762 amostras de mitigação geradas a partir das trigger words do modelo oficial treinado com `train_augmented_original.json`:
- **`train_clean_mof_semi_rp.json`** (76.300 amostras)
- **`MOF_semi_rp.json`** (762 amostras)
- **`train_augmented_mof_semi_rp.json`** (76.735 amostras)

### Experimento 4: `train_clean_mof_oficial` (Basic + MOF scratch retrain)
- **Dataset**: `MOF/train_clean_mof_semi_rp.json` (76.300 amostras)
- **Modo**: Scratch Retrain (`--save_step 100 --warmup 100 --epochs 3`)

In [ ]:
# Célula 6: Treinamento 4 - train_clean_mof_oficial
run_mof_experiment(
    name="train_clean_mof_oficial",
    train_set="MOF/train_clean_mof_semi_rp.json",
    save_step=100,
    warmup=100,
    epochs=3,
    resume_ckpt=None
)


### Experimento 5: `train_augmented_mof_finetuning_oficial` (Basic + Aug + MOF finetuning)
- **Dataset**: `MOF/MOF_semi_rp.json` (762 amostras)
- **Modo**: Fine-Tuning com `--resume` a partir do checkpoint de `train_augmented_original`
- **Safeguard Anti-Colapso**: `--save_step 2 --warmup 2` e validação imediata no step 1

In [ ]:
# Célula 7: Treinamento 5 - train_augmented_mof_finetuning_oficial
base_ckpt = find_checkpoint([
    f"{DRIVE_OUTPUT_DIR}/train_augmented_original/best_model.pth",
    "checkpoints/train_augmented_original/best_model.pth",
    f"{DRIVE_OUTPUT_DIR}/train_augmented_rp/best_model.pth"
])
print(f"Utilizando checkpoint base: {base_ckpt}")

run_mof_experiment(
    name="train_augmented_mof_finetuning_oficial",
    train_set="MOF/MOF_semi_rp.json",
    save_step=2,
    warmup=2,
    epochs=3,
    resume_ckpt=base_ckpt
)


### Experimento 6: `train_augmented_mof_oficial` (Basic + Aug + MOF scratch retrain)
- **Dataset**: `MOF/train_augmented_mof_semi_rp.json` (76.735 amostras)
- **Modo**: Scratch Retrain (`--save_step 100 --warmup 100 --epochs 3`)

In [ ]:
# Célula 8: Treinamento 6 - train_augmented_mof_oficial
run_mof_experiment(
    name="train_augmented_mof_oficial",
    train_set="MOF/train_augmented_mof_semi_rp.json",
    save_step=100,
    warmup=100,
    epochs=3,
    resume_ckpt=None
)


## Vertente 3: Vertente Reproduzida (Re-gerada via Trigger Words Reproduzidas)
Nesta vertente, utilizamos as 762 amostras de mitigação geradas a partir das trigger words do modelo reproduzido treinado com `train_augmented_rp.json`:
- **`train_clean_mof_rp.json`** (76.300 amostras)
- **`MOF_rp.json`** (762 amostras)
- **`train_augmented_mof_rp.json`** (76.735 amostras)

### Experimento 7: `train_clean_mof` (Basic + MOF scratch retrain)
- **Dataset**: `MOF/train_clean_mof_rp.json` (76.300 amostras)
- **Modo**: Scratch Retrain (`--save_step 100 --warmup 100 --epochs 3`)

In [ ]:
# Célula 9: Treinamento 7 - train_clean_mof
run_mof_experiment(
    name="train_clean_mof",
    train_set="MOF/train_clean_mof_rp.json",
    save_step=100,
    warmup=100,
    epochs=3,
    resume_ckpt=None
)


### Experimento 8: `train_augmented_mof_finetuning` (Basic + Aug + MOF finetuning)
- **Dataset**: `MOF/MOF_rp.json` (762 amostras)
- **Modo**: Fine-Tuning com `--resume` a partir do checkpoint de `train_augmented_rp` (reproduzido)
- **Safeguard Anti-Colapso**: `--save_step 2 --warmup 2` e validação imediata no step 1

In [ ]:
# Célula 10: Treinamento 8 - train_augmented_mof_finetuning
base_ckpt = find_checkpoint([
    f"{DRIVE_OUTPUT_DIR}/train_augmented_rp/best_model.pth",
    "checkpoints/train_augmented_rp/best_model.pth"
])
print(f"Utilizando checkpoint base: {base_ckpt}")

run_mof_experiment(
    name="train_augmented_mof_finetuning",
    train_set="MOF/MOF_rp.json",
    save_step=2,
    warmup=2,
    epochs=3,
    resume_ckpt=base_ckpt
)


### Experimento 9: `train_augmented_mof` (Basic + Aug + MOF scratch retrain)
- **Dataset**: `MOF/train_augmented_mof_rp.json` (76.735 amostras)
- **Modo**: Scratch Retrain (`--save_step 100 --warmup 100 --epochs 3`)

In [ ]:
# Célula 11: Treinamento 9 - train_augmented_mof
run_mof_experiment(
    name="train_augmented_mof",
    train_set="MOF/train_augmented_mof_rp.json",
    save_step=100,
    warmup=100,
    epochs=3,
    resume_ckpt=None
)


### Desligamento Automático do Runtime (Google Colab)
Libera imediatamente os recursos de computação desconectando a GPU via `runtime.unassign()`.
Isso assegura que não haverá consumo desnecessário de créditos de GPU após o término dos treinamentos.

In [ ]:
# Célula 12: Verificação Final e Desligamento da VM
import time
import os

print("\n" + "=" * 75)
print("🎉 Todos os treinamentos da Fase 2 (MOF) foram processados!")
print("📁 Status dos 9 Modelos no Google Drive:")

all_present = True
for exp in EXPERIMENT_NAMES:
    path = f"{DRIVE_OUTPUT_DIR}/{exp}/best_model.pth"
    present = os.path.exists(path)
    if not present:
        all_present = False
    status = "OK (Salvo)" if present else "PENDENTE/AUSENTE"
    print(f"   [{status:15}] {exp}")

print("=" * 75)
print("⚡ Desconectando o runtime da VM para economizar créditos de GPU...")

try:
    from google.colab import runtime
    time.sleep(5)
    runtime.unassign()
except Exception as e:
    print(f"Não foi possível desconectar automaticamente via runtime.unassign(): {e}")
